# INNER JOIN: pairing rows on a key

Practice notebook for the datafromzero SQL lesson **INNER JOIN: pairing rows on a key** (S19). The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Predict, then count the pairs

In [ ]:
con.sql("""
SELECT COUNT(*) AS joined_rows
FROM order_items AS oi
INNER JOIN products AS p ON oi.product_id = p.product_id
""").show()

### Step 2. Look at the pairs

In [ ]:
con.sql("""
SELECT oi.order_item_id, oi.order_id, p.product_name,
       oi.quantity, p.unit_price_usd, oi.discount_usd
FROM order_items AS oi
INNER JOIN products AS p ON oi.product_id = p.product_id
ORDER BY oi.order_item_id
LIMIT 5
""").show()

### Step 3. Compute the line amount

In [ ]:
con.sql("""
SELECT oi.order_item_id,
       p.product_name,
       oi.quantity * p.unit_price_usd - oi.discount_usd AS line_amount_usd
FROM order_items AS oi
INNER JOIN products AS p ON oi.product_id = p.product_id
ORDER BY oi.order_item_id
LIMIT 5
""").show()

### Step 4. Add it up

In [ ]:
con.sql("""
SELECT SUM(oi.quantity * p.unit_price_usd - oi.discount_usd) AS sales_usd
FROM order_items AS oi
INNER JOIN products AS p ON oi.product_id = p.product_id
""").show()

### Step 5. Group by product

In [ ]:
con.sql("""
SELECT p.product_name,
       SUM(oi.quantity) AS units,
       SUM(oi.quantity * p.unit_price_usd - oi.discount_usd) AS sales_usd
FROM order_items AS oi
INNER JOIN products AS p ON oi.product_id = p.product_id
GROUP BY p.product_name
ORDER BY sales_usd DESC
""").show()

### Step 6. Join in the other direction

In [ ]:
con.sql("""
SELECT COUNT(*) AS joined_rows
FROM orders AS o
INNER JOIN order_items AS oi ON o.order_id = oi.order_id
""").show()

### Step 7. Verify

## Exercise 1. What did completed orders sell, in dollars?

Given: the grain is a line, and the first part of the query is step 4's. The status lives in `orders`, so add a second `INNER JOIN`, to `orders` on `order_id`, and keep only the completed orders. E21 reported Sales of 22,749.25: your result should match it.

In [ ]:
con.sql("""
SELECT SUM(oi.quantity * p.unit_price_usd - oi.discount_usd) AS sales_usd
FROM order_items AS oi
INNER JOIN products AS p ON oi.product_id = p.product_id
-- add the join to orders and the filter on completed orders here
""").show()

## Exercise 2. Orders joined to customers

A teammate says: "orders joined to customers gives one row per order, so 1,800 rows." Before you run anything, write one sentence in the cell below: how many rows will the join of `orders` to `customers` on `customer_id` return, and why? Then run it, run a plain `COUNT(*)` of `orders`, and write one sentence: where did the difference go?

In [ ]:
# My expected result, in words:
#

In [ ]:
# Query 1: orders INNER JOIN customers, counted

In [ ]:
# Query 2: the count of orders

## Exercise 3. Spot the flaw

A colleague wants each order next to its lines and runs this query. It gives an error, not a table. Read the error, then fix the query.

In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    con.sql("""
    SELECT order_id, status, quantity
    FROM orders AS o
    INNER JOIN order_items AS oi ON o.order_id = oi.order_id
    """).show()
except Exception as e:
    print(e)


In [ ]:
# Your corrected query: